# Multivariate Forecasting: Manufacturing Energy & Sensor Signals
# 多变量预测：制造业能耗与传感器信号

This notebook uses an industrial manufacturing scenario where power consumption is affected by production volume, machine temperature, vibration, and line speed.

本教程使用制造业场景：产线能耗受产量、机器温度、振动和线速影响。

Covered API:

- `feature_cols` for multi-input forecasting
- multi-output `target_col=[...]`
- `itransformer` and `srs_net` model names
- `ModelPipeline` with multivariate NN models

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(7)
n = 240
date = pd.date_range("2023-01-01", periods=n, freq="h")
shift = ((date.hour >= 8) & (date.hour < 20)).astype(int)
production = 80 + 25 * shift + 10 * np.sin(np.linspace(0, 8*np.pi, n)) + rng.normal(0, 4, n)
line_speed = 1.0 + 0.15 * shift + rng.normal(0, 0.03, n)
temperature = 45 + 0.08 * production + 5 * np.sin(np.linspace(0, 4*np.pi, n)) + rng.normal(0, 1.5, n)
vibration = 0.4 + 0.006 * production + rng.normal(0, 0.05, n)
power_kw = 120 + 1.7 * production + 16 * line_speed + 0.9 * temperature + rng.normal(0, 8, n)

plant = pd.DataFrame({
    "date": date,
    "power_kw": power_kw,
    "production_units": production,
    "temperature_c": temperature,
    "vibration": vibration,
    "line_speed": line_speed,
    "shift": shift,
})
plant.head()

In [ ]:
from PipelineTS.pipeline import ModelPipeline

feature_cols = ["power_kw", "production_units", "temperature_c", "vibration", "line_speed", "shift"]
train, valid = plant.iloc[:-24].copy(), plant.iloc[-24:].copy()

available = ModelPipeline.list_all_available_models()
multivariate_models = [m for m in ["itransformer", "srs_net"] if m in available]
multivariate_models

In [ ]:
if multivariate_models:
    selected_mv_models = multivariate_models[:1]
    mv_kwargs = {}
    if "itransformer" in selected_mv_models:
        mv_kwargs.update({"itransformer__epochs": 30, "itransformer__patience": 6})
    if "srs_net" in selected_mv_models:
        mv_kwargs.update({"srs_net__epochs": 30, "srs_net__patience": 6})
    pipe_miso = ModelPipeline(
        time_col="date",
        target_col="power_kw",
        feature_cols=feature_cols,
        lags=24,
        include_models=selected_mv_models,
        quantile=None,
        cv=2,
        **mv_kwargs,
    )
    lb_miso = pipe_miso.fit(train, valid_data=valid)
    display(lb_miso)
    display(pipe_miso.predict(24).head())
else:
    print("No multivariate NN backend is available in this environment.")

In [ ]:
targets = ["power_kw", "temperature_c"]
feature_cols_mimo = ["power_kw", "temperature_c", "production_units", "vibration", "line_speed", "shift"]

if "itransformer" in available:
    pipe_mimo = ModelPipeline(
        time_col="date",
        target_col=targets,
        feature_cols=feature_cols_mimo,
        lags=24,
        include_models=["itransformer"],
        quantile=None,
        cv=2,
        itransformer__epochs=30,
        itransformer__patience=6,
    )
    pipe_mimo.fit(train, valid_data=valid)
    pipe_mimo.predict(12).head()

## Practical guidance / 实践建议

- Use `feature_cols` when drivers are observed historically and should be used as model inputs.
- Use `known_covariates` when future values are known at prediction time, such as calendar, planned promotions, planned shifts, or prices.
- Use `target_col=[...]` only when you need simultaneous forecasting of multiple business KPIs.